# 1. Import Library
Memasukkan semua library yang dibutuhkan untuk tahap EDA, Preprocessing, Modeling, dan Evaluasi.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

import warnings
warnings.filterwarnings('ignore')

# 2. EDA & Preprocessing

### Load Dataset
Sesuai ketentuan, data **train** hanya digunakan untuk training (termasuk sebagai patokan imputasi dan scaling), dan data **test** khusus digunakan untuk evaluasi akhir. Tidak ada split data lagi di sini.

*(Catatan: Untuk menghindari RAM laptop penuh / Kernel Crash akibat memuat 7,5 juta baris sekaligus, kita langsung mensampling data saat awal load)*

In [2]:
# Mengambil data yang sudah di-split sebelumnya
train_df = pd.read_csv('../dataset/data/train.csv')
test_df = pd.read_csv('../dataset/data/test.csv')

# ================== MEMORY OPTIMIZATION ==================
# Agar laptop tidak crash (Out of Memory), kita ambil 
# 300.000 sampel acak untuk train, dan 50.000 untuk test.
train_limit = min(300000, len(train_df))
test_limit = min(50000, len(test_df))
train_df = train_df.sample(n=train_limit, random_state=42).reset_index(drop=True)
test_df = test_df.sample(n=test_limit, random_state=42).reset_index(drop=True)
# =========================================================

print("Shape of train_df:", train_df.shape)
print("Shape of test_df:", test_df.shape)
train_df.head()

Shape of train_df: (300000, 46)
Shape of test_df: (50000, 46)


,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-5220458,Source1,2,2022-08-26 16:45:19,2022-08-27 17:43:40,36.147841,-95.965453,36.147836,-95.964697,0.042,...,False,False,True,False,False,False,Day,Day,Day,Day
1,A-910157,Source2,2,2021-09-07 05:26:59,2021-09-07 07:18:00,35.031857,-78.875946,NaN,NaN,0.000,...,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3898300,Source1,2,2022-06-21 17:31:50,2022-06-21 18:52:31,29.930587,-90.079140,29.931243,-90.079700,0.056,...,False,False,False,False,False,False,Day,Day,Day,Day
3,A-7633280,Source1,4,2018-01-16 04:13:45,2018-01-16 10:13:45,40.840531,-73.281480,40.837230,-73.281200,0.229,...,False,False,False,False,False,False,Night,Night,Night,Night
4,A-3183102,Source2,2,2017-11-13 09:07:30,2017-11-13 09:37:14,31.693895,-106.333054,NaN,NaN,0.000,...,False,False,False,False,False,False,Day,Day,Day,Day


### 2.1 Missing Value Handling
Penanganan missing value dilakukan dengan mengisi kekosongan (imputasi). Kolom numerik diisi dengan **median**, sedangkan kolom kategorikal diisi dengan **modus**. Ingat, perhitungan median/modus **hanya** didapatkan dari data train.

In [3]:
print("Missing values di data train:\n", train_df.isnull().sum())

# Tentukan mana kolom numerik dan kategorikal
numeric_cols = train_df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = train_df.select_dtypes(include=['object', 'category']).columns

# Imputasi Numeric: Menggunakan median dari train_df
for col in numeric_cols:
    median_val = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_val)
    test_df[col] = test_df[col].fillna(median_val)  # Terapkan median ke test_df

# Imputasi Categorical: Menggunakan modus dari train_df
for col in categorical_cols:
    mode_val = train_df[col].mode()[0]
    train_df[col] = train_df[col].fillna(mode_val)
    test_df[col] = test_df[col].fillna(mode_val)

Missing values di data train:
 ID                            0
Source                        0
Severity                      0
Start_Time                    0
End_Time                      0
Start_Lat                     0
Start_Lng                     0
End_Lat                  131936
End_Lng                  131936
Distance(mi)                  0
Description                   0
Street                      431
City                          8
County                        0
State                         0
Zipcode                      70
Country                       0
Timezone                    280
Airport_Code                830
Weather_Timestamp          4634
Temperature(F)             6301
Wind_Chill(F)             77756
Humidity(%)                6692
Pressure(in)               5452
Visibility(mi)             6810
Wind_Direction             6778
Wind_Speed(mph)           22061
Precipitation(in)         85681
Weather_Condition          6695
Amenity                       0
Bump     

### 2.2 Outliers Handling
Kita menggunakan metode IQR (Interquartile Range) untuk membatasi (capping) nilai outlier agar tidak merusak model. PENTING: Kolom target dilarang keras untuk dicapping!

In [4]:
for col in numeric_cols:
    if col == 'Severity':
        continue # Lewati kolom target, jangan di-capping!
        
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Terapkan batas bawah dan atas ke train_df
    train_df[col] = np.where(train_df[col] < lower_bound, lower_bound, train_df[col])
    train_df[col] = np.where(train_df[col] > upper_bound, upper_bound, train_df[col])
    
    # Terapkan batas dari train ke test_df
    test_df[col] = np.where(test_df[col] < lower_bound, lower_bound, test_df[col])
    test_df[col] = np.where(test_df[col] > upper_bound, upper_bound, test_df[col])

### 2.3 Encoding
Mengubah data string (teks) menjadi angka numerik yang bisa dipahami model menggunakan `LabelEncoder`. Dibuat secepat mungkin menggunakan dictionary mapping.

In [5]:
for col in categorical_cols:
    le = LabelEncoder()
    # Belajar pattern dari train
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    
    # Membuat kamus (dictionary) mapping agar pencarian secepat kilat
    le_dict = dict(zip(le.classes_, range(len(le.classes_))))
    unknown_val = len(le.classes_) # Angka baru untuk kategori yang belum dikenal
    
    # Tambahkan class unknown
    le.classes_ = np.append(le.classes_, '<unknown>')
    
    # Transform test_df dengan map (jauh lebih cepat dari lambda)
    test_df[col] = test_df[col].astype(str).map(le_dict).fillna(unknown_val).astype(int)

### 2.4 Transformasi (Scaling Fitur)
Melakukan standardisasi (rata-rata=0, std=1) agar model yang sensitif pada jarak bekerja lebih optimal.

In [6]:
# TARGET KOLOM DISET KE 'Severity'
TARGET_KOLOM = 'Severity' 

try:
    # Memisahkan Fitur (X) dan Target Label (y)
    X_train_raw = train_df.drop(columns=[TARGET_KOLOM])
    y_train = train_df[TARGET_KOLOM]
    
    X_test_raw = test_df.drop(columns=[TARGET_KOLOM])
    y_test = test_df[TARGET_KOLOM]
    
    # Inisiasi Scaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled = scaler.transform(X_test_raw) # Scaling test pakai patokan train
except KeyError:
    print("WARNING: Jangan lupa ubah variabel TARGET_KOLOM di atas dengan nama kolom yang benar di dataset ini.")

# 3. Modeling
Membuat model klasifikasi. Kita akan menggunakan algoritma **Random Forest Classifier** karena secara umum algoritma ini tangguh (robust) terhadap struktur data yang belum linear sempurna dan memberikan akurasi yang solid.

In [7]:
try:
    # Menggunakan class_weight='balanced' untuk mengatasi imbalanced data (kelas 1 dan 4 yang langka)
    # Karena di Langkah 2 data sudah di-sample, datanya sudah teracak dengan baik.
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced')
    
    print(f"Melatih model Random Forest (Balanced) pada seluruh train_df ({len(X_train_scaled)} baris)...")
    rf_model.fit(X_train_scaled, y_train)
    print("Model Random Forest berhasil dilatih!")
except NameError:
    print("Harap lengkapi tahap 2.4 terlebih dahulu.")

Melatih model Random Forest (Balanced) pada seluruh train_df (300000 baris)...
Model Random Forest berhasil dilatih!


# 4. Evaluasi (Test)
Menggunakan fitur dari data uji (test.csv) ke dalam model, lalu membandingkan hasil prediksinya dengan label aslinya untuk mencari nilai Recall, Precision, dan Accuracy.

In [8]:
try:
    print(f"Menguji data test...")
    y_pred = rf_model.predict(X_test_scaled)
    
    # Evaluasi Metriks
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print("=== HASIL EVALUASI MODEL ===")
    print(f"Akurasi   (Accuracy)  : {acc:.4f}")
    print(f"Presisi   (Precision) : {prec:.4f}")
    print(f"Recall    (Recall)    : {rec:.4f}")
    
    # Menampilkan report lebih lengkap (opsional)
    print("\nClassification Report Lengkap:\n")
    print(classification_report(y_test, y_pred, zero_division=0))
except NameError:
    print("Harap lengkapi tahap sebelumnya.")

Menguji data test...
=== HASIL EVALUASI MODEL ===
Akurasi   (Accuracy)  : 0.8359
Presisi   (Precision) : 0.8133
Recall    (Recall)    : 0.8359

Classification Report Lengkap:

              precision    recall  f1-score   support

           1       0.00      0.00      0.00       415
           2       0.84      0.98      0.91     39928
           3       0.75      0.32      0.45      8326
           4       0.60      0.00      0.00      1331

    accuracy                           0.84     50000
   macro avg       0.55      0.33      0.34     50000
weighted avg       0.81      0.84      0.80     50000

